# USIP (Underwater Sonar Intelligence Platform) — Colab Training Bridge

This Google Colab notebook provides **GPU-accelerated model training** for the USIP prototype:
1. **Known Target Fine-Tuning** (Shipwrecks, Pipelines, Debris/Objects) using YOLOv8/YOLOv11.
2. **Acoustic Anomaly Encoder / VAE Training** on unlabelled sonar background patches.
3. **Export Weights** directly back to your local repository or Google Drive.

In [ ]:
# 1. Verify GPU Acceleration
!nvidia-smi
import torch
print(f"PyTorch Version: {torch.__version__}")
print(f"CUDA Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"Active Device: {torch.cuda.get_device_name(0)}")
else:
    print("WARNING: Running on CPU. Please switch Colab Runtime to T4 GPU (Runtime -> Change runtime type -> T4 GPU).")

In [ ]:
# 2. Install High-Performance Sonar ML Dependencies
!pip install -q ultralytics albumentations opencv-python-headless timm

In [ ]:
# 3. Mount Google Drive (Optional: to access datasets or save trained models)
from google.colab import drive
import os

drive.mount('/content/drive')
DRIVE_WORKSPACE = '/content/drive/MyDrive/USIP_Models'
os.makedirs(DRIVE_WORKSPACE, exist_ok=True)
print(f"Checkpoint directory ready at: {DRIVE_WORKSPACE}")

### 4. Known Target Detector Training (YOLOv8)
We configure the 3 target classes: `Shipwreck`, `Pipeline`, `Debris`.

In [ ]:
import yaml

data_yaml = {
    'path': '/content/dataset',
    'train': 'images/train',
    'val': 'images/val',
    'names': {
        0: 'Shipwreck',
        1: 'Pipeline',
        2: 'Debris'
    }
}

with open('/content/usip_sonar.yaml', 'w') as f:
    yaml.dump(data_yaml, f)

print("Sonar dataset configuration created at /content/usip_sonar.yaml")

In [ ]:
from ultralytics import YOLO

# Load lightweight YOLOv8 nano / small pretrained on underwater/visual features
model = YOLO('yolov8n.pt')

# Train with Sonar-tailored Hyperparameters
# - Single channel/grayscale augmentations
# - Reduced color jitter, increased scale/translation
print("Ready to initiate training on GPU:")
# results = model.train(
#     data='/content/usip_sonar.yaml',
#     epochs=50,
#     imgsz=640,
#     batch=16,
#     device=0,
#     name='usip_detector_v1'
# )

### 5. Unsupervised Anomaly VAE & SSL Training on Unlabelled Sonar Data
Trains directly on unlabelled sonar seabed patches from `sss_ssl_dataset_N713_384`.
Normal seafloor achieves low reconstruction loss and dense clustering, while novel targets produce distinct high-anomaly signatures.

In [ ]:
# Extract unlabelled SSS SSL patches from multi-volume archive in Colab
!apt-get install -y -q p7zip-full
# If sss_ssl_dataset is uploaded to Google Drive:
# !7z x /content/drive/MyDrive/sss_ssl_dataset_N713_384.zip -o/content/unlabelled_sonar/ -y
print("7-Zip ready to extract unlabelled sonar multi-volume dataset.")

In [ ]:
import torch
import torch.nn as nn

class SonarPatchAutoencoder(nn.Module):
    def __init__(self, latent_dim=64):
        super().__init__()
        # Encoder
        self.encoder = nn.Sequential(
            nn.Conv2d(1, 32, 4, stride=2, padding=1), # 32x32
            nn.ReLU(),
            nn.Conv2d(32, 64, 4, stride=2, padding=1), # 16x16
            nn.ReLU(),
            nn.Conv2d(64, 128, 4, stride=2, padding=1), # 8x8
            nn.ReLU(),
            nn.Flatten(),
            nn.Linear(128 * 8 * 8, latent_dim)
        )
        # Decoder
        self.decoder = nn.Sequential(
            nn.Linear(latent_dim, 128 * 8 * 8),
            nn.Unflatten(1, (128, 8, 8)),
            nn.ConvTranspose2d(128, 64, 4, stride=2, padding=1),
            nn.ReLU(),
            nn.ConvTranspose2d(64, 32, 4, stride=2, padding=1),
            nn.ReLU(),
            nn.ConvTranspose2d(32, 1, 4, stride=2, padding=1),
            nn.Sigmoid()
        )

    def forward(self, x):
        z = self.encoder(x)
        recon = self.decoder(z)
        return recon, z

ae = SonarPatchAutoencoder().cuda() if torch.cuda.is_available() else SonarPatchAutoencoder()
print("Sonar Patch Autoencoder instantiated:", ae)

### 6. Export Trained Models for Local USIP Dashboard
Run this cell after training finishes to save weights to Google Drive.

In [ ]:
!cp -r runs/detect/usip_detector_v1/weights/best.pt /content/drive/MyDrive/USIP_Models/usip_detector.pt 2>/dev/null || echo "Train a run first to copy."
print("Trained model saved. You can place 'usip_detector.pt' into 'backend/models/' in your local workspace!")